In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/qa-retrieval-rag-chunking/__results__.html
/kaggle/input/qa-retrieval-rag-chunking/index.faiss
/kaggle/input/qa-retrieval-rag-chunking/__huggingface_repos__.json
/kaggle/input/qa-retrieval-rag-chunking/__notebook__.ipynb
/kaggle/input/qa-retrieval-rag-chunking/__output__.json
/kaggle/input/qa-retrieval-rag-chunking/custom.css
/kaggle/input/zalo-dataset/corpus.csv
/kaggle/input/zalo-dataset/crawled_test_5k.jsonl


In [2]:
!pip install protobuf==3.20.3
!pip install sentence-transformers==4.1.0 
!pip install faiss-gpu-cu12==1.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-me

In [3]:
import torch
import pandas as pd
import numpy as np
import faiss
import json
import os
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm.auto import tqdm
import gc

2026-01-01 14:48:23.264487: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767278903.416896      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767278903.460283      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
import torch
import pandas as pd
import numpy as np
import faiss
import json
import os
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm.auto import tqdm
import gc

def evaluate_retrieval_with_json(
    test_data_path, 
    corpus_data_path, 
    faiss_index_path, 
    cross_encoder_name='BAAI/bge-reranker-v2-m3'
    ):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # --- 1. Load Models ---
    print("\n--- Loading Models ---")
    bi_encoder = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder', device=device)
    if device == 'cuda':
        bi_encoder.half()

    print(f"Loading Cross-Encoder: {cross_encoder_name}")
    cross_encoder = CrossEncoder(cross_encoder_name, max_length=512, device=device)
    if device == 'cuda':
        cross_encoder.model.half()

    # --- 2. Load Corpus Data, FAISS Index, and Prepare Corpus IDs ---
    print("\n--- Loading Corpus Data & FAISS Index ---")
    print(f"Loading corpus data from: {corpus_data_path}")
   
    df_chunks = pd.read_csv(corpus_data_path, encoding='utf-8')
    
    # Kiểm tra tên cột để tránh lỗi (ưu tiên law_id/article_id theo dataset mới)
    df_chunks['law_id_norm'] = df_chunks['law_id'].astype(str).str.lower().str.replace("đ", "d")
    df_chunks['article_id_norm'] = df_chunks['article_id'].fillna('').astype(str)
    df_chunks['unique_id'] = df_chunks['law_id_norm'] + "_" + df_chunks['article_id_norm']
    
    all_chunk_ids = df_chunks['unique_id'].values
    all_chunk_texts = ("Title: " + df_chunks['title'].astype(str) + ". Content: " + df_chunks['text'].astype(str)).values

    print(f"Loading FAISS index from: {faiss_index_path}")
    index = faiss.read_index(faiss_index_path)

    # --- 3. Load and Process Test Data from JSONL ---
    print(f"\n--- Loading and Processing Test Data from: {test_data_path} ---")
    
    queries = []
    ground_truths = []
    
    with open(test_data_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            
            # Đọc từng dòng (Line-delimited JSON)
            item = json.loads(line)
            
            # Lấy thông tin theo cấu trúc của Zalo dataset
            q = item.get("question", "").strip()
            rels = item.get("relevant_articles", [])
            
            if q and rels:
                queries.append(q)
                
                # Xử lý Ground Truths cho query này
                gt_set = set()
                for vb in rels:
                    # Chuẩn hóa: lower -> replace 'đ' -> 'd'
                    # Notebook mẫu dùng key: law_id, article_id
                    l_id = vb.get('law_id', vb.get('law_id', ''))
                    a_id = vb.get('article_id', vb.get('article_id', ''))
                    
                    lh = str(l_id).lower().replace("đ", "d")
                    dieu = str(a_id)
                    
                    if lh and dieu:
                        gt_set.add(f"{lh}_{dieu}")
                
                ground_truths.append(list(gt_set))
        
    total_queries = len(queries)
    print(f"Loaded {total_queries} queries from JSONL file.")

    # --- 4. Prepare for Evaluation ---
    k_max = 10
    recall_scores_retriever = {1: 0, 5: 0, 10: 0}
    recall_scores_reranker = {1: 0, 5: 0, 10: 0}
    
    # --- 5. Stage 1: Retrieval (Bi-Encoder + FAISS) ---
    print(f"\n--- Stage 1: Retrieval using Bi-Encoder ---")
    print(f"Encoding {total_queries} test queries...")
    query_embeddings = bi_encoder.encode(
        queries, convert_to_numpy=True, show_progress_bar=True, batch_size=512
    )
    
    query_embeddings_float32 = query_embeddings.astype('float32')
    faiss.normalize_L2(query_embeddings_float32)

    print(f"Performing FAISS search for top {k_max} documents...")
    _, top_k_indices = index.search(query_embeddings_float32, k_max)

    # --- 6. Stage 2: Reranking and Evaluation Loop ---
    print(f"\n--- Stage 2: Reranking and Evaluation ---")
    for i in tqdm(range(total_queries), desc="Retrieving, Reranking, and Evaluating"):
        query = queries[i]
        ground_truth_id_set = set(ground_truths[i])

        if not ground_truth_id_set:
            continue

        # Get initial retrieved docs from FAISS results
        retrieved_indices = top_k_indices[i]
        valid_indices = [idx for idx in retrieved_indices if idx != -1]
        
        # Kiểm tra biên để tránh lỗi index out of bounds nếu index faiss cũ hơn corpus
        retrieved_doc_ids = []
        retrieved_doc_texts = []
        for idx in valid_indices:
            if idx < len(all_chunk_ids):
                retrieved_doc_ids.append(all_chunk_ids[idx])
                retrieved_doc_texts.append(all_chunk_texts[idx])

        # Evaluate Retriever Recall
        for k_val in recall_scores_retriever.keys():
            if not ground_truth_id_set.isdisjoint(retrieved_doc_ids[:k_val]):
                recall_scores_retriever[k_val] += 1
        
        # Rerank the retrieved documents
        if not retrieved_doc_texts:
            continue 

        sentence_pairs = [[query, doc] for doc in retrieved_doc_texts]
        scores = cross_encoder.predict(sentence_pairs, show_progress_bar=False)

        # Sort results by the new scores
        ranked_results = sorted(zip(scores, retrieved_doc_ids), key=lambda x: x[0], reverse=True)
        reranked_doc_ids = [doc_id for score, doc_id in ranked_results]

        # Evaluate Reranker Recall
        for k_val in recall_scores_reranker.keys():
            if not ground_truth_id_set.isdisjoint(reranked_doc_ids[:k_val]):
                recall_scores_reranker[k_val] += 1

    # --- 7. Display Final Results ---
    print("\n--- Final Evaluation Results ---")

    print("\n--- Before Reranking (Bi-Encoder Retrieval) ---")
    for k_val in sorted(recall_scores_retriever.keys()):
        recall = recall_scores_retriever[k_val] / total_queries if total_queries > 0 else 0
        print(f"Recall@{k_val}: {recall:.4f}")

    print("\n--- After Reranking (Cross-Encoder) ---")
    for k_val in sorted(recall_scores_reranker.keys()):
        recall = recall_scores_reranker[k_val] / total_queries if total_queries > 0 else 0
        print(f"Recall@{k_val}: {recall:.4f}")

In [5]:
if __name__ == "__main__":
    BASE_INPUT_DIR = '/kaggle/input'
    TEST_DATA_PATH = os.path.join(BASE_INPUT_DIR, 'zalo-dataset/crawled_test_5k.jsonl')
    CORPUS_DATA_PATH = os.path.join(BASE_INPUT_DIR, 'zalo-dataset/corpus.csv')
    FAISS_INDEX_PATH = os.path.join(BASE_INPUT_DIR, 'qa-retrieval-rag-chunking/index.faiss')
    evaluate_retrieval_with_json(test_data_path=TEST_DATA_PATH, corpus_data_path=CORPUS_DATA_PATH, faiss_index_path=FAISS_INDEX_PATH)

Using device: cuda

--- Loading Models ---


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading FAISS index from: /kaggle/input/qa-retrieval-rag-chunking/index.faiss

--- Loading and Processing Test Data from: /kaggle/input/zalo-dataset/crawled_test_5k.jsonl ---
Loaded 5000 queries from JSONL file.

--- Stage 1: Retrieval using Bi-Encoder ---
Encoding 5000 test queries...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Performing FAISS search for top 10 documents...

--- Stage 2: Reranking and Evaluation ---


Retrieving, Reranking, and Evaluating:   0%|          | 0/5000 [00:00<?, ?it/s]


--- Final Evaluation Results ---

--- Before Reranking (Bi-Encoder Retrieval) ---
Recall@1: 0.3984
Recall@5: 0.6464
Recall@10: 0.7256

--- After Reranking (Cross-Encoder) ---
Recall@1: 0.5684
Recall@5: 0.7132
Recall@10: 0.7256
